In [1]:
# Configuration AWS
BEDROCK_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
AWS_REGION = "us-east-1"

In [6]:
# Imports
import os
import boto3

from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings
from langchain_aws import ChatBedrock
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

In [17]:
# Structured JSON System Prompt

SYSTEM_PROMPT = """
You are an advanced Machine Learning Assistant specializing in extracting and explaining technical content from PDFs. 
You must provide comprehensive, detailed explanations while maintaining perfect JSON structure with markdown-formatted content.

Your response MUST be a valid JSON object with this exact structure:

{
  "title": "Clear, descriptive title based on the user query and PDF content",
  "content": "DETAILED explanation with markdown formatting (see requirements below)",
  "summary": "Comprehensive 4-5 sentence summary covering key concepts",
  "key_concepts": [
    "List of 5-7 most important concepts with brief explanations",
    "Each concept should be self-contained and informative"
  ],
  "mathematical_expressions": {
    "formulas": [
      {
        "name": "Formula name",
        "expression": "$$formula in LaTeX$$",
        "description": "Explanation of what the formula represents and when to use it"
      }
    ],
    "variables": [
      {
        "symbol": "Variable symbol",
        "definition": "What the variable represents",
        "units": "Units of measurement if applicable"
      }
    ]
  },
  "definitions": {
    "term": "Detailed definition with context and examples"
  },
  "examples": [
    {
      "scenario": "Example scenario description",
      "application": "How the concept applies here",
      "calculation": "$$relevant formula or calculation$$"
    }
  ],
  "quiz": {
    "topic": "Quiz topic based on content",
    "questions": [
      {
        "question": "Question text",
        "options": ["A) ...", "B) ...", "C) ...", "D) ..."],
        "correct_answer": "A",
        "explanation": "Brief explanation of why this answer is correct"
      }
    ]
  },
  "code_snippets": [
    {
      "language": "python/r/etc.",
      "purpose": "What this code demonstrates",
      "code": "```language\\nactual code here\\n```"
    }
  ],
  "figures_and_tables": {
    "description": "Description of any important figures/tables from PDF",
    "insights": "Key insights derived from visual data"
  }
}

# CONTENT REQUIREMENTS:

## Detailed Explanations:
- Provide THOROUGH, in-depth explanations of every concept mentioned
- Break down complex ideas into digestible parts
- Include real-world applications and examples
- Connect concepts to show relationships and dependencies
- Explain the 'why' behind formulas and methods, not just the 'what'

## Markdown Formatting for 'content' field:
Use these markdown elements to structure the explanation:

### Headers:
# Main Topic
## Subtopic
### Key Point

### Text Formatting:
- **Bold** for emphasis on key terms
- *Italic* for introducing new concepts
- `inline code` for functions, variables, or commands

### Lists:
- Bullet points for lists of items
- Numbered steps for procedures
  - Nested items for sub-points

### Code Blocks:
```python
# For code examples
def example_function():
    return "code here"
```

### Mathematical Expressions:
- Block equations: $$frac{d}{dx}e^x = e^x$$
- Inline equations: $\alpha + \beta = \gamma$

### Tables:
| Parameter | Value | Description |
|-----------|-------|-------------|
| Example   | 42    | Description here |

### Blockquotes:
> Important notes or definitions
> that need special emphasis

### Links:
[Reference to section](#section-name)

### Horizontal Rules:
--- for separating major sections

## Formula Handling:
- Extract ALL mathematical expressions from the PDF
- Provide both LaTeX notation AND plain-text explanation
- Explain each variable and its significance
- Include derivations or proofs if present in the PDF
- Show example calculations with the formula

## Figures and Visual Content:
- Describe any graphs, charts, or diagrams in detail
- Explain trends, patterns, and relationships shown
- Interpret the significance of visual data
- Note any axes labels, scales, or legends

## Examples and Applications:
- Provide at least 2-3 concrete examples
- Show step-by-step solutions where applicable
- Include both simple and complex applications
- Connect examples to real-world scenarios

## Code Implementation:
- Include practical code snippets demonstrating concepts
- Explain what each code block accomplishes
- Show expected output or results
- Provide comments within code for clarity

## Definitions and Terminology:
- Define ALL technical terms
- Include alternative names or synonyms
- Provide context for when terms are used
- Note any domain-specific usage

# IMPORTANT RULES:

1. **JSON Structure**: Output ONLY valid JSON - no explanations outside the JSON structure
2. **Content Source**: Base ALL content on the retrieved PDF context and user query
3. **Completeness**: Extract and explain EVERYTHING relevant from the PDF
4. **Missing Information**: If information is not found, use empty arrays [] or empty strings ""
5. **Accuracy**: Ensure all mathematical expressions and technical details are correct
6. **Depth**: Provide comprehensive explanations - assume the user wants to deeply understand the topic
7. **Organization**: Structure content logically from basic to advanced concepts
8. **Markdown Integration**: Use markdown liberally within string fields to enhance readability
9. **Escape Characters**: Properly escape all double quotes and backslashes for valid JSON
10. **LaTeX Formatting**: Use $$ for display equations and $ for inline equations

# SPECIAL HANDLING:

## For Mathematical Heavy Content:
- Extract formulas into the mathematical_expressions section
- Explain derivations step-by-step in the content section
- Include variable definitions with units
- Show example calculations

## For Algorithm Explanations:
- Provide pseudocode in code_snippets
- Explain time/space complexity
- Show input/output examples
- Discuss edge cases and limitations

## For Theoretical Concepts:
- Provide intuitive explanations first
- Follow with rigorous mathematical formulation
- Include historical context if relevant
- Discuss implications and applications

## For Practical Applications:
- Include hands-on examples
- Provide implementation guidance
- Discuss common pitfalls
- Show performance considerations

Remember: Your goal is to provide a COMPREHENSIVE understanding of the PDF content. The user should feel they have learned everything important from the document after reading your response.
"""

print(" Enhanced system prompt configured for detailed JSON output with markdown support")


 Enhanced system prompt configured for detailed JSON output with markdown support


<>:103: SyntaxWarning: invalid escape sequence '\g'
<>:103: SyntaxWarning: invalid escape sequence '\g'
C:\Users\ACER\AppData\Local\Temp\ipykernel_1780\1302663958.py:103: SyntaxWarning: invalid escape sequence '\g'
  - Inline equations: $\alpha + \beta = \gamma$


In [12]:
# AGENT SETUP

FAISS_PATH = "faiss_index"

# Initialize Embeddings
embedding = BedrockEmbeddings(
    model_id=BEDROCK_EMBEDDING_MODEL_ID,
    region_name=AWS_REGION
)

# Initialize Vector Store
vector_store = FAISS.load_local(
    FAISS_PATH,
    embedding,
    allow_dangerous_deserialization=True
)

# Create Retriever with top-k search
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)


# PDF Retriever Tool
@tool
def retrieve_context(query: str) -> str:
    """Search technical PDF documents for relevant information about Machine Learning."""
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join(d.page_content for d in docs)

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Perform a web search for the latest information on Machine Learning."""
    return search.invoke(query)

# Bedrock LLM 
bedrock_llm = ChatBedrock(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    model_kwargs={
        "max_tokens": 10000,
        "temperature": 0.4,
        "top_p": 0.9
    }
)

# Create Agent
agent = create_agent(
    model=bedrock_llm,
    tools=[retrieve_context, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("Agent initialized successfully!")

Agent initialized successfully!


In [13]:
query  = """

# 1. Introduction to Unsupervised Learning

Machine learning algorithms are broadly divided into two categories:

1. **Supervised Learning**
2. **Unsupervised Learning**

In **supervised learning**, the dataset contains **labels** that indicate the expected output.

In **unsupervised learning**, the dataset **does not contain labels**, and the algorithm must discover patterns on its own.

### Definition

Unsupervised learning is a group of machine learning techniques used to **identify patterns, structures, or relationships in unlabeled data**.

The goal is to analyze the dataset and uncover meaningful insights without predefined outcomes.

### Common Unsupervised Learning Algorithms

Some commonly used unsupervised learning methods include:

* Clustering
* Neural networks (certain unsupervised variants)
* Anomaly detection

---

# 2. Real-World Example of Unsupervised Learning

A common real-world example is **Google News article grouping**.

Google News collects thousands of articles from different sources and automatically groups similar articles together.

### How does it work?

1. The system analyzes words and phrases in each article.
2. It calculates similarities between articles based on frequent terms.
3. Articles with similar word patterns are grouped together.

This grouping process uses **clustering**, which is a type of unsupervised learning.

---

# 3. Labeled vs Unlabeled Data

Understanding the difference between labeled and unlabeled data is essential.

### Unlabeled Data

Example dataset:

```
Point 1: (1,2)
Point 2: (2,2)
Point 3: (3,1)
```

These points only contain coordinates and no category information.

### Labeled Data

Example dataset:

```
Point 1: (1,2) -> Danger Zone
Point 2: (2,2) -> Normal Zone
Point 3: (3,1) -> Normal Zone
```

In labeled data, each datapoint already belongs to a known category.

Unsupervised learning works primarily with **unlabeled datasets**.

---

# 4. What is Clustering?

Clustering is one of the most widely used unsupervised learning techniques.

### Definition

Clustering is the process of **grouping data points with similar characteristics together**.

The main idea is:

* Points within the same cluster are **more similar to each other**
* Points from different clusters are **less similar**

A common way to measure similarity is **distance** between points in space.

For example, in a 2D coordinate plane:

* Points close together belong to the same cluster
* Points far apart belong to different clusters

---

# 5. Visualizing Data for Clustering

Clustering often starts with **visualizing data points**.

Example dataset representing hypothetical **Pokemon sightings**:

```python
from matplotlib import pyplot as plt

x_coordinates = [80, 93, 86, 98, 86, 9, 15, 3, 10, 20, 44, 56, 49, 62, 44]
y_coordinates = [87, 96, 95, 92, 92, 57, 49, 47, 59, 55, 25, 2, 10, 24, 10]

plt.scatter(x_coordinates, y_coordinates)
plt.show()
```

This scatter plot visualizes the distribution of points on a 2D plane.

When plotted, the points naturally form **groups**, which represent clusters.

---

# 6. What is a Cluster?

A cluster is simply:

> A group of items with similar characteristics.

### Examples of clusters

1. **Google News**

   * Articles with similar words appear in the same cluster.

2. **Customer segmentation**

   * Customers with similar purchasing behavior form a cluster.

Clustering helps identify hidden patterns in data.

---

# 7. Types of Clustering Algorithms

There are several clustering algorithms available.

The most common ones are:

### 1. Hierarchical Clustering

Creates a tree-like structure of clusters.

Clusters are formed by repeatedly merging or splitting groups based on similarity.

### 2. K-Means Clustering

One of the most popular clustering algorithms.

It divides the dataset into **K predefined clusters**.

### Other Clustering Algorithms

* DBSCAN
* Gaussian Mixture Models

Each algorithm has its own strengths and use cases.

---

# 8. Hierarchical Clustering in Python

Hierarchical clustering can be implemented using **SciPy**.

Example code:

```python
from scipy.cluster.hierarchy import linkage, fcluster
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

x_coordinates = [80.1,93.1,86.6,98.5,86.4,9.5,15.2,3.4,10.4,20.3,44.2,56.8,49.2,62.5,44.0]
y_coordinates = [87.2,96.1,95.6,92.4,92.4,57.7,49.4,47.3,59.1,55.5,25.6,2.1,10.9,24.1,10.3]

df = pd.DataFrame({
    'x_coordinate': x_coordinates,
    'y_coordinate': y_coordinates
})

Z = linkage(df, 'ward')

df['cluster_labels'] = fcluster(Z, 3, criterion='maxclust')

sns.scatterplot(
    x='x_coordinate',
    y='y_coordinate',
    hue='cluster_labels',
    data=df
)

plt.show()
```

This code:

1. Creates a dataset
2. Performs hierarchical clustering
3. Assigns cluster labels
4. Visualizes clusters on a scatter plot

---

# 9. K-Means Clustering in Python

K-Means clustering can also be implemented using SciPy.

Example:

```python
from scipy.cluster.vq import kmeans, vq
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import random

random.seed((1000,2000))

x_coordinates = [80.1,93.1,86.6,98.5,86.4,9.5,15.2,3.4,10.4,20.3,44.2,56.8,49.2,62.5,44.0]
y_coordinates = [87.2,96.1,95.6,92.4,92.4,57.7,49.4,47.3,59.1,55.5,25.6,2.1,10.9,24.1,10.3]

df = pd.DataFrame({
    'x_coordinate': x_coordinates,
    'y_coordinate': y_coordinates
})

centroids, _ = kmeans(df, 3)

df['cluster_labels'], _ = vq(df, centroids)

sns.scatterplot(
    x='x_coordinate',
    y='y_coordinate',
    hue='cluster_labels',
    data=df
)

plt.show()
```

### How K-Means Works

1. Choose **K cluster centers (centroids)**
2. Assign each point to the nearest centroid
3. Recalculate centroid positions
4. Repeat until clusters stabilize

---

# 10. Data Preparation for Clustering

Before performing clustering, data preparation is essential.

### Why Data Preparation is Important

Real-world data often has issues such as:

1. Different measurement units

   * Example: product size in cm, price in dollars

2. Different value scales

   * Example: travel expenses vs cereal purchases

3. Dominance of one variable

   * Large-scale features can bias clustering results

To address these issues, we use **normalization**.

---

# 11. Data Normalization

Normalization rescales variables so that they have **comparable magnitudes**.

### Formula

```
x_new = x / std_dev(x)
```

This transforms data so its **standard deviation becomes 1**.

### Example in Python

```python
from scipy.cluster.vq import whiten

data = [5,1,3,3,2,3,3,8,1,2,2,3,5]

scaled_data = whiten(data)

print(scaled_data)
```

Output:

```
[2.73, 0.55, 1.64, 1.64, 1.09, 1.64, 1.64, 4.36, 0.55, 1.09, 1.09, 1.64, 2.73]
```

Normalization ensures that **all variables contribute equally** to clustering.

---

# 12. Visualizing Normalized Data

You can visualize the difference between original and scaled data.

Example:

```python
from matplotlib import pyplot as plt

plt.plot(data, label="original")
plt.plot(scaled_data, label="scaled")

plt.legend()
plt.show()
```

This plot compares the original data values with normalized values.

---

# Conclusion

Cluster analysis is an essential technique in unsupervised learning used to discover patterns in unlabeled datasets.

Key takeaways:

* Unsupervised learning works without labeled data.
* Clustering groups similar items together.
* Common clustering methods include:

  * Hierarchical clustering
  * K-Means clustering
* Data normalization is crucial for accurate clustering results.

Clustering has applications in many domains, including:

* News article grouping
* Customer segmentation
* Recommendation systems
* Anomaly detection

"""

# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [15]:
import json
from pprint import pprint

# Get the AIMessage object
ai_msg = response["messages"][1]

# Extract JSON string
json_str = ai_msg.content
print("Raw JSON string from AI response:"
      )
print(json_str)
# Convert string into dict
# data = json.loads(json_str)

# Pretty print
# pprint(data, width=120)

Raw JSON string from AI response:
{
  "title": "Introduction to Unsupervised Learning and Clustering",
  "content": "# 1. Introduction to Unsupervised Learning\n\nMachine learning algorithms are broadly divided into two categories:\n\n1. **Supervised Learning**\n2. **Unsupervised Learning**\n\nIn **supervised learning**, the dataset contains **labels** that indicate the expected output.\n\nIn **unsupervised learning**, the dataset **does not contain labels**, and the algorithm must discover patterns on its own.\n\n### Definition\n\nUnsupervised learning is a group of machine learning techniques used to **identify patterns, structures, or relationships in unlabeled data**.\n\nThe goal is to analyze the dataset and uncover meaningful insights without predefined outcomes.\n\n### Common Unsupervised Learning Algorithms\n\nSome commonly used unsupervised learning methods include:\n\n* Clustering\n* Neural networks (certain unsupervised variants)\n* Anomaly detection\n\n---\n\n# 2. Real-Worl

In [16]:
# Extract JSON string
mdata = ai_msg.additional_kwargs

# Pretty print
pprint(mdata, width=120)

{'model_id': 'anthropic.claude-3-haiku-20240307-v1:0',
 'stop_reason': 'max_tokens',
 'usage': {'cache_read_input_tokens': 0,
           'cache_write_input_tokens': 0,
           'completion_tokens': 4096,
           'prompt_tokens': 4266,
           'total_tokens': 8362}}
